# Gathering Relevant Data

In [ ]:
# Base Directory
library(here)
BASE_DIR <- here()
print(BASE_DIR)

# Other dependencies
library(dplyr)

[1] "/Users/amberteetsel/MSDS/STAT_5010/opioid-crisis"


## NCHS - Drug Poisoning Mortality by State

In [62]:
# API Call
nchs_df <- read.csv(here("data", "NCHS_Mortality_Clean.csv"))
head(nchs_df)
dim(nchs_df)

# Keep relevant columns
nchs_df = nchs_df[, c("state", "year", "sex", "age_group", "age_group_detail", "race", "population", "death_rate")]
head(nchs_df)
dim(nchs_df)

,state,year,sex,age_group,age_group_detail,race,death_rate,death_rate_log,us_crude_rate,us_age_adjusted_rate,deaths,population
,<chr>,<int>,<chr>,<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<int>,<int>
1,Alabama,1999,Both Sexes,All Ages,All Ages,All Races,3.8521,0.6859297,6.0382,6.0570,169,4430143
2,Alabama,2000,Both Sexes,All Ages,All Ages,All Races,4.4857,0.7392321,6.1882,6.1749,197,4447100
3,Alabama,2001,Both Sexes,All Ages,All Ages,All Races,4.8915,0.7702259,6.8057,6.7922,216,4467634
4,Alabama,2002,Both Sexes,All Ages,All Ages,All Races,4.7619,0.7605657,8.1766,8.1957,211,4480089
5,Alabama,2003,Both Sexes,All Ages,All Ages,All Races,4.4333,0.7350637,8.8881,8.8765,197,4503491
6,Alabama,2004,Both Sexes,All Ages,All Ages,All Races,6.3542,0.8665354,9.3660,9.3831,283,4530729


[1] 2862   12

,state,year,sex,age_group,age_group_detail,race,population,death_rate
,<chr>,<int>,<chr>,<chr>,<chr>,<chr>,<int>,<dbl>
1,Alabama,1999,Both Sexes,All Ages,All Ages,All Races,4430143,3.8521
2,Alabama,2000,Both Sexes,All Ages,All Ages,All Races,4447100,4.4857
3,Alabama,2001,Both Sexes,All Ages,All Ages,All Races,4467634,4.8915
4,Alabama,2002,Both Sexes,All Ages,All Ages,All Races,4480089,4.7619
5,Alabama,2003,Both Sexes,All Ages,All Ages,All Races,4503491,4.4333
6,Alabama,2004,Both Sexes,All Ages,All Ages,All Races,4530729,6.3542


[1] 2862    8

In [63]:
# Filter for Entire US so we can use demographic breakdown
nchs_us <- nchs_df %>%
    filter(
        state == "United States",
        sex != "Both Sexes",
        age_group != "All Ages",
        race != "All Races",
        year >= 2000
    )
nchs_us = nchs_us[, c("year", "sex", "age_group", "age_group_detail", "race", "death_rate")]

# Filter for State-Level Data
nchs_state <- nchs_df %>%
    filter(
        state != "United States",
        sex == "Both Sexes",
        age_group == "All Ages",
        race == "All Races",
        year >= 2000
    )

## DEA: Retail Drug Transactions

In [38]:
# Load Data
dea_df <- read.csv(here("data", "DEA_Clean.csv"))
head(dea_df)

,state,year,hydro_gms,oxy_gms,fent_gms
,<chr>,<int>,<dbl>,<dbl>,<dbl>
1,AK,2000,27018.40,74395.72,613.2600
2,AK,2001,32213.77,110300.16,613.2600
3,AK,2002,35524.00,106095.60,722.8675
4,AK,2003,38834.23,101891.03,832.4750
5,AK,2004,42144.46,97686.46,942.0825
6,AK,2005,45454.69,93481.90,1051.6900


In [40]:
# Aggregate to US Level
dea_us = dea_df %>%
    group_by(year) %>%
    summarise(
        hydro_gms = sum(hydro_gms, na.rm=TRUE),
        oxy_gms = sum(oxy_gms, na.rm = TRUE),
        fent_gms = sum(fent_gms, na.rm = TRUE)
    )
head(dea_us)

year,hydro_gms,oxy_gms,fent_gms
<int>,<dbl>,<dbl>,<dbl>
2000,14530961,15233461,185509.8
2001,15555319,19851864,185509.8
2002,18118626,22518655,235870.3
2003,20681932,25185446,286230.9
2004,23245239,27852236,336591.5
2005,25808546,30519027,386952.1


## University of Kentucky Center for Poverty Research (UKCPR) National Welfare Data

In [67]:
uk_raw <- read.csv(here("data", "ukcpr_raw.csv"))
head(uk_raw)

,state_name,state,state_fips,year,Population,Employment,Unemployment,Unemployment.rate,Marginally.Food.Insecure,Food.Insecure,⋯,SSI.recipients..Disabled,Medicaid.beneficiaries,WIC.participation,NSLP.Free.Participation,NSLP.Reduced.Participation,NSLP.Total.Participation,SBP.Free.Participation,SBP.Reduced.Participation,SBP.Total.Participation,Fraction.of.the.Year.ACA.Expansion
,<chr>,<int>,<int>,<int>,<int>,<int>,<int>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,AL,1,1,1980,3893888,1521183,148106,8.9,NA,NA,⋯,NA,NA,NA,NA,NA,NA,NA,NA,NA,0
2,AK,2,2,1980,401851,169397,18008,9.6,NA,NA,⋯,NA,NA,NA,NA,NA,NA,NA,NA,NA,0
3,AZ,3,4,1980,2718215,1146371,81630,6.6,NA,NA,⋯,NA,NA,NA,NA,NA,NA,NA,NA,NA,0
4,AR,4,5,1980,2286435,922894,75386,7.6,NA,NA,⋯,NA,NA,NA,NA,NA,NA,NA,NA,NA,0
5,CA,5,6,1980,23667902,10787673,791379,6.8,NA,NA,⋯,NA,NA,NA,NA,NA,NA,NA,NA,NA,0
6,CO,6,8,1980,2889964,1405381,86267,5.8,NA,NA,⋯,NA,NA,NA,NA,NA,NA,NA,NA,NA,0


In [68]:
uk_raw <- uk_raw %>%
    filter(
        year >= 2000 & year <= 2016,
    )
uk_raw <- uk_raw[, c("state_name", "year", "Population", "Unemployment",
    "Number.of.Poor..thousands.", "Gross.State.Product", "Food.Stamp.SNAP.Recipients",
    "Medicaid.beneficiaries", "State.Minimum.Wage")]

uk_raw <- uk_raw %>% 
    rename(
        state = state_name,
        year = year,
        pop = Population,
        unempl_pop = Unemployment,
        poverty_pop = Number.of.Poor..thousands.,
        gsp = Gross.State.Product,
        snap_pop = Food.Stamp.SNAP.Recipients,
        medicaid_pop = Medicaid.beneficiaries,
        min_wage = State.Minimum.Wage
    )

head(uk_raw)

,state,year,pop,unempl_pop,poverty_pop,gsp,snap_pop,medicaid_pop,min_wage
,<chr>,<int>,<int>,<int>,<int>,<dbl>,<dbl>,<int>,<dbl>
1,AL,2000,4452173,97629,583,120132.9,396057,564926,5.15
2,AK,2000,627963,20365,47,26815.8,37524,83161,5.65
3,AZ,2000,5160586,99302,607,164609.9,259006,483993,5.15
4,AR,2000,2678588,53606,436,68740.4,246572,392018,5.15
5,CA,2000,33987977,834629,4294,1366166.5,1831698,6168816,6.25
6,CO,2000,4326921,65107,425,180693.3,155948,278969,5.15


In [ ]:
# Calculate Rates for State-Level Data
uk_state <- uk_raw %>%
    summarize(
        unempl_rate = (unempl_pop / pop) * 100000,
        poverty_rate = (poverty_pop / pop) * 100000,
        poverty_rate = (poverty_pop / pop) * 100000,
    )

In [55]:
# Aggregate to the U.S. level, weighted averages
uk_us <- df %>%
    group_by(year) %>%
    summarize(
        state = "United States",
        total_pop = sum(pop, na.rm = TRUE),
        unempl_rate = (sum(unempl_pop, na.rm = TRUE) / total_pop) * 100000,
        poverty_rate = (sum(poverty_pop, na.rm = TRUE) / total_pop * 100000),
        snap_rate = (sum(snap_pop, na.rm = TRUE) / total_pop) * 100000,
        medicaid_rate = (sum(medicaid_pop, na.rm = TRUE) / total_pop) * 100000,
        min_wage = sum(min_wage * pop, na.rm=TRUE) / total_pop,
        gsp = sum(gsp, na.rm = TRUE)
    )

uk_us <- uk_us[, c("year", "unempl_rate", "poverty_rate", "snap_rate", "medicaid_rate", "min_wage", "gsp")]
head(uk_us)

year,unempl_rate,poverty_rate,snap_rate,medicaid_rate,min_wage,gsp
<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
2000,2019.001,11.19320,6060.809,12264.16,5.111411,10186440
2001,2389.221,11.54828,6062.912,13108.12,5.335078,10515541
2002,2914.736,12.01807,6625.599,14295.28,5.407161,10868827
2003,3004.228,12.36092,7315.337,15138.72,5.427446,11385473
2004,2770.054,12.63468,8134.589,15552.35,5.453500,12137766
2005,2570.526,12.50420,8691.976,15906.85,5.616907,12957341


In [66]:
# Join Data on Year
df_final = nchs_us %>%
    left_join(dea_us, by = "year") %>%
    left_join(uk_us, by = "year")

head(df_final)

,year,sex,age_group,age_group_detail,race,death_rate,hydro_gms,oxy_gms,fent_gms,unempl_rate,poverty_rate,snap_rate,medicaid_rate,min_wage,gsp
,<int>,<chr>,<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,2016,Male,0-14,0–14,Hispanic,0.1527,293948813,20385937192,810267655.2,2391.576,12.57721,13669.539,23024.97,8.164271,18631950
2,2012,Female,0-14,0–14,Hispanic,0.0686,1729331353,948160309,985067403.9,3988.981,14.81530,14829.803,19005.70,7.450122,16083771
3,2000,Female,0-14,0–14,Hispanic,0.0584,14530961,15233461,185509.8,2019.001,11.19320,6060.809,12264.16,5.111411,10186440
4,2001,Female,0-14,0–14,Hispanic,0.0930,15555319,19851864,185509.8,2389.221,11.54828,6062.912,13108.12,5.335078,10515541
5,2002,Female,0-14,0–14,Hispanic,0.1256,18118626,22518655,235870.3,2914.736,12.01807,6625.599,14295.28,5.407161,10868827
6,2003,Female,0-14,0–14,Hispanic,0.1562,20681932,25185446,286230.9,3004.228,12.36092,7315.337,15138.72,5.427446,11385473
